# Autogen
AutoGen is an open‑source framework for building collaborative, multi‑agent AI systems that use LLM‑driven agents to communicate and solve complex tasks autonomously.

From an architectural perspective, AutoGen agents themselves are built around:
User Proxy Agent — Represents the human or system initiating a request
Assistant Agent — The main reasoning and problem‑solving agent.
Tool/Function Agent — Executes specialized operations such as API calls, computation, or data retrieval.

AutoGen Tax Filing Assistant
==============================
A multi-agent AI application for tax computation and filing guidance.

Agents:
  1. TaxpayerProxy       – Represents the user; collects income/deduction inputs
  2. TaxComputationAgent – Computes tax liability (India IT slabs + US placeholders)
  3. TaxAdvisoryAgent    – Suggests deductions, exemptions, and filing strategies
  4. ComplianceAgent     – Validates data, flags errors, and guides ITR form selection
  5. OrchestratorAgent   – Routes messages and summarises the final report

In [ ]:
import autogen
import os, json
from autogen import register_function

In [ ]:
apikey = os.environ['OPENAI_API_KEY']

# --------------------------
# Example 1) Getting Started
# --------------------------

# LLM configuration
# Creates a configuration dictionary for the LLM (Large Language Model).
llm_config = {"model": "gpt-4o",  "api_key": apikey, "temperature": 0 }

In [ ]:
def compute_india_tax(gross_income: float, regime: str = "new") -> dict:
    """
    Compute Indian income tax for FY 2024-25.
    regime: 'new' (default) or 'old'
    """

    if regime == "new":
        std_deduction = 75_000
        taxable = max(0, gross_income - std_deduction)

        # New Regime slabs (FY 2024-25)
        if taxable <= 300_000:
            tax = 0
        elif taxable <= 600_000:
            tax = (taxable - 300_000) * 0.05
        elif taxable <= 900_000:
            tax = (300_000 * 0.05) + (taxable - 600_000) * 0.10
        elif taxable <= 1_200_000:
            tax = (300_000 * 0.05) + (300_000 * 0.10) + (taxable - 900_000) * 0.15
        elif taxable <= 1_500_000:
            tax = (300_000 * 0.05) + (300_000 * 0.10) + (300_000 * 0.15) + (taxable - 1_200_000) * 0.20
        else:
            tax = (300_000 * 0.05) + (300_000 * 0.10) + (300_000 * 0.15) + (300_000 * 0.20) + (taxable - 1_500_000) * 0.30

    else:  # Old Regime
        std_deduction = 50_000
        taxable = max(0, gross_income - std_deduction)

        # Old Regime slabs
        if taxable <= 250_000:
            tax = 0
        elif taxable <= 500_000:
            tax = (taxable - 250_000) * 0.05
        elif taxable <= 1_000_000:
            tax = (250_000 * 0.05) + (taxable - 500_000) * 0.20
        else:
            tax = (250_000 * 0.05) + (500_000 * 0.20) + (taxable - 1_000_000) * 0.30

    # Rebate u/s 87A — New regime: up to ₹25,000 rebate if taxable income ≤ ₹7L
    rebate = 0.0
    if regime == "new" and taxable <= 700_000:
        rebate = min(tax, 25_000)

    # Surcharge
    surcharge = 0.0
    if taxable > 10_000_000:
        surcharge = tax * 0.15
    elif taxable > 5_000_000:
        surcharge = tax * 0.10

    # Health & Education Cess @ 4%
    cess = (tax + surcharge) * 0.04

    net_tax = max(0, tax + surcharge + cess - rebate)

    return {
        "gross_income": gross_income,
        "standard_deduction": std_deduction,
        "taxable_income": taxable,
        "base_tax": round(tax, 2),
        "rebate_87A": round(rebate, 2),
        "surcharge": round(surcharge, 2),
        "cess_4pct": round(cess, 2),
        "total_tax_payable": round(net_tax, 2),
        "effective_rate_pct": round((net_tax / gross_income) * 100, 2) if gross_income else 0,
        "regime": regime,
    }

In [ ]:
# No need to execute this
# income_details = compute_india_tax(750000)

In [ ]:
def suggest_deductions(income_details: dict) -> dict:
    """
    Suggest common Indian IT deductions based on the income profile.
    income_details keys: has_home_loan, has_80C_investments, has_health_insurance,
                         has_hra, has_nps, donations
    """
    suggestions = []
    max_savings = 0.0

    if income_details.get("has_80C_investments"):
        suggestions.append({
            "section": "80C",
            "description": "ELSS, PPF, EPF, LIC, NSC, 5-yr FD, tuition fees",
            "max_deduction": 150_000,
        })
        max_savings += 150_000

    if income_details.get("has_health_insurance"):
        suggestions.append({
            "section": "80D",
            "description": "Health insurance premium (self + family + parents)",
            "max_deduction": 75_000,
        })
        max_savings += 75_000

    if income_details.get("has_home_loan"):
        suggestions.append({
            "section": "24(b)",
            "description": "Home loan interest deduction",
            "max_deduction": 200_000,
        })
        max_savings += 200_000

    if income_details.get("has_nps"):
        suggestions.append({
            "section": "80CCD(1B)",
            "description": "Additional NPS contribution over 80C limit",
            "max_deduction": 50_000,
        })
        max_savings += 50_000

    if income_details.get("has_hra"):
        suggestions.append({
            "section": "HRA Exemption",
            "description": "House Rent Allowance — exempt based on actual rent, salary, city",
            "max_deduction": "Varies",
        })

    if income_details.get("donations"):
        suggestions.append({
            "section": "80G",
            "description": "Donations to eligible charitable institutions (50%–100% deduction)",
            "max_deduction": "Varies",
        })

    return {
        "applicable_deductions": suggestions,
        "estimated_max_additional_savings_INR": max_savings,
        "note": "Available primarily under Old Tax Regime; 80CCD(1B) also available under New Regime",
    }


In [ ]:
def recommend_itr_form(income_type: str, total_income: float, is_company: bool = False) -> dict:
    """Recommend the correct ITR form for the taxpayer."""
    
    if is_company:
        form = "ITR-6"
        reason = "Companies (other than those claiming exemption u/s 11)"
    elif income_type == "salary" and total_income <= 5_000_000:
        form = "ITR-1 (Sahaj)"
        reason = "Resident individuals with salary, income ≤ ₹50L"
    elif income_type == "salary":
        form = "ITR-2"
        reason = "Individuals/HUF with salary + capital gains or income > ₹50L"
    elif income_type in ("business", "profession"):
        form = "ITR-3"
        reason = "Individuals/HUF with business or professional income"
    elif income_type == "presumptive":
        form = "ITR-4 (Sugam)"
        reason = "Presumptive income under sections 44AD, 44ADA, 44AE"
    else:
        form = "ITR-2"
        reason = "Default for individuals with capital gains or foreign income"

    return {
        "recommended_form": form,
        "reason": reason,
        "due_date": "31st July (non-audit) / 31st October (audit cases)",
        "e_filing_portal": "https://www.incometax.gov.in/iec/foportal",
    }

In [ ]:
def validate_tax_data(data: dict) -> dict:
    """Validate mandatory fields and flag common errors."""
    errors = []
    warnings = []

    if not data.get("pan"):
        errors.append("PAN (Permanent Account Number) is mandatory.")
    elif len(data["pan"]) != 10:
        errors.append("PAN must be exactly 10 characters.")

    if not data.get("aadhaar"):
        warnings.append("Aadhaar linking is mandatory for e-filing.")

    if data.get("gross_income", 0) < 0:
        errors.append("Gross income cannot be negative.")

    if data.get("tds_deducted", 0) > data.get("gross_income", float("inf")):
        warnings.append("TDS deducted exceeds gross income — verify Form 26AS.")

    if not data.get("bank_account"):
        errors.append("At least one pre-validated bank account is needed for refund.")

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings,
        "action": "Resolve errors before filing; address warnings where possible.",
    }

# pip install ag2[openai]

In [ ]:
# ******************************************************
# Assistant agent
# Creates AI assistant agent that will answer questions.
# *******************************************************

# compute_india_tax
tax_computation_agent = autogen.AssistantAgent(name="TaxComputationAgent", llm_config=llm_config, 
                                               system_message="Compute Indian income tax for FY 24-25")

# suggest_deductions
tax_advisory_agent = autogen.AssistantAgent(name="TaxAdvisoryAgent",llm_config=llm_config, 
                                            system_message=''' Suggest common Indian IT deductions based on the income profile.
    income_details keys: has_home_loan, has_80C_investments, has_health_insurance, has_hra, has_nps, donations''')

# validate_tax_data
compliance_agent = autogen.AssistantAgent(name="ComplianceAgent", llm_config=llm_config, 
    system_message="""Validate mandatory fields and flag common errors.""")


# **********************************************************
# User proxy agent
# Creates a user-side agent that represents the human.
# This agent acts like you (the user) in the conversation.

# Code_execution = False: Disabling code execution.
# Agent will:
#   NOT run Python code
#   ONLY chat with the assistant

# **********************************************************
user_proxy = autogen.UserProxyAgent( name="TaxpayerProxy", code_execution_config=False,human_input_mode="TERMINATE")

In [ ]:
# ── After your agent definitions ────────────────────────────────

# STEP 1: Register functions (after agents are defined)

register_function(compute_india_tax, caller=tax_computation_agent, executor=user_proxy, 
                  description="Compute Indian income tax for FY 2024-25 under new and old regime.")

register_function(suggest_deductions, caller=tax_advisory_agent, executor=user_proxy,
    description="Suggest applicable Indian IT deductions based on taxpayer profile.")

register_function(recommend_itr_form, caller=compliance_agent, executor=user_proxy,
    description="Recommend the correct ITR form based on income type and total income." )

register_function( validate_tax_data, caller=compliance_agent, executor=user_proxy,
    description="Validate taxpayer data for completeness before filing." )

In [ ]:
# STEP 2: Group Chat — all agents talk to each other
group_chat = autogen.GroupChat(
    agents=[user_proxy, tax_computation_agent, tax_advisory_agent, compliance_agent],
    messages=[], 
    max_round=15,
    speaker_selection_method="auto",   # GPT-4 decides who speaks next
)


# STEP 3: Group Chat Manager — drives the conversation
manager = autogen.GroupChatManager( groupchat=group_chat, llm_config=llm_config,)

In [ ]:
# STEP 4: Taxpayer scenario — the input to kick off the session
TAXPAYER_SCENARIO = """
I need help with my Income Tax Filing for FY 2024-25. Here are my details:

Name        : Rahul Sharma
PAN         : ABCPS1234R
Aadhaar     : Linked
Age         : 35, Resident Individual

Income:
- Gross Salary : ₹18,00,000
- HRA Received : ₹2,40,000 (paying rent ₹25,000/month)
- Interest Income (savings) : ₹12,000
- TDS already deducted : ₹1,85,000

Investments & Expenses:
- PPF : ₹50,000
- ELSS Mutual Funds : ₹50,000
- LIC Premium : ₹25,000
- Health Insurance : ₹22,000
- Home Loan Interest : ₹1,80,000
- NPS Tier-1 : ₹50,000
- Bank account for refund : Yes

Please:
1. Compute tax under both old and new regime
2. Suggest all applicable deductions
3. Recommend the best regime and correct ITR form
4. Validate my data and guide me through e-filing
"""

In [ ]:
# STEP 5: Start the conversation
if __name__ == "__main__":
    # print("=" * 60)
    # print("   AutoGen Tax Filing Assistant  |  FY 2024-25")
    # print("=" * 60)

    tax_result = user_proxy.initiate_chat(manager, message=TAXPAYER_SCENARIO, )

In [ ]:
tax_result

In [ ]:
type(tax_result)

In [ ]:
for msg in tax_result.chat_history:
    print("ROLE:", msg.get("role"))
    print("NAME:", msg.get("name"))
    print("CONTENT:", msg.get("content"))
    print("-" * 50)